# Baseline CVAE Test

In [9]:
from pathlib import Path
import pickle

import torch
import sys, os

project_root = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)


from baseline.training import train_model
from baseline.model import PeptideCVAE
from baseline.inference import generate_peptides

In [14]:
# Paths relative to this notebook
baseline_dir = Path(".")
dataset_file = baseline_dir / "../database/training_data.json"
model_path = baseline_dir / "cvae_peptide_model.pth"
scaler_path = baseline_dir / "scaler.pkl"

# Quick 1-epoch train
model, device = train_model(
    dataset_file=str(dataset_file),
    epochs=50,
    batch_size=32,
    max_len=12,
    latent_dim=32,
    model_path=str(model_path),
)

print(f"Model saved at: {model_path}")
print(f"Scaler expected at: {scaler_path}")

Using device: cuda
Scaler saved to scaler.pkl
Epoch [1/50] | Beta: 0.00 | Train Loss: 21.3240 | Train Recon: 21.3240 | Train KL: 65.2862 | Val Loss: 17.3760 | Val Recon: 17.3760 | Val KL: 119.8356
Epoch [5/50] | Beta: 0.16 | Train Loss: 14.7255 | Train Recon: 13.0589 | Train KL: 10.4162 | Val Loss: 14.3840 | Val Recon: 12.6735 | Val KL: 10.6905
Epoch [10/50] | Beta: 0.36 | Train Loss: 13.1158 | Train Recon: 10.5568 | Train KL: 7.1085 | Val Loss: 13.6636 | Val Recon: 11.1486 | Val KL: 6.9863
Epoch [15/50] | Beta: 0.56 | Train Loss: 12.8750 | Train Recon: 10.0116 | Train KL: 5.1133 | Val Loss: 13.6646 | Val Recon: 10.9207 | Val KL: 4.8997
Epoch [20/50] | Beta: 0.76 | Train Loss: 12.5463 | Train Recon: 9.9645 | Train KL: 3.3971 | Val Loss: 13.6288 | Val Recon: 11.0105 | Val KL: 3.4451
Epoch [25/50] | Beta: 0.96 | Train Loss: 12.0046 | Train Recon: 10.0711 | Train KL: 2.0140 | Val Loss: 13.6430 | Val Recon: 11.8291 | Val KL: 1.8895
Epoch [30/50] | Beta: 1.00 | Train Loss: 11.0216 | Train R

In [15]:
# Reload model + scaler and generate a few peptides
with open(scaler_path, "rb") as f:
    scaler = pickle.load(f)

# Keep architecture args aligned with training defaults
loaded_model = PeptideCVAE(
    vocab_size=24,
    condition_dim=8,
    max_seq_len=14,
    latent_dim=32,
)
loaded_model.load_state_dict(torch.load(model_path, map_location=device))

# Trying to imitate this:
# {
#     "dbaasp_id": "DBAASPS_373",
#     "sequence": "KLFKRWKHLFR",
#     "length": 11,
#     "smiles": "CC(C)C[C@H](NC(=O)[C@H](Cc1cnc[nH]1)NC(=O)[C@H](CCCCN)NC(=O)[C@H](Cc1c[nH]c2ccccc12)NC(=O)[C@H](CCCN=C(N)N)NC(=O)[C@H](CCCCN)NC(=O)[C@H](Cc1ccccc1)NC(=O)[C@H](CC(C)C)NC(=O)[C@@H](N)CCCCN)C(=O)N[C@@H](Cc1ccccc1)C(=O)N[C@@H](CCCN=C(N)N)C(=O)O",
#     "ph": null,
#     "molecular_weight": 1558.9480000000003,
#     "logp": -0.992100000000006,
#     "net_charge": 5.0,
#     "isoelectric_point": 12.18,
#     "hydrophobicity": 1.05,
#     "cathionicity": 6,
#     "target_groups": [
#         "Gram+"
#     ],
#     "complexity": "Monomer"
# }
target = [11, 7, 1558.94, -0.9921, 5.0, 12.18, 1.05, 6]

samples = generate_peptides(
    model=loaded_model,
    scaler=scaler,
    num_samples=5,
    properties=target,
    temperature=0.9,
    top_k=5,
)

samples

Using device: cuda
Model loaded successfully.

Target Properties:
- Length: 11.0
- pH: 7.0
- Molecular Weight: 1558.94
- LogP: -0.9921
- Net Charge: 5.0
- Isoelectric Point: 12.18
- Hydrophobicity: 1.05
- Cathionicity: 6.0

--- Generating 5 novel peptide(s) ---
Sample 1: FLKWLKHKKFW
Sample 2: KWIKRFLRKWR
Sample 3: KRILKHFWKYF
Sample 4: KRIWKYLLKHK
Sample 5: KRIWKIYKRFK


['FLKWLKHKKFW', 'KWIKRFLRKWR', 'KRILKHFWKYF', 'KRIWKYLLKHK', 'KRIWKIYKRFK']